<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

In [4]:
from torch import randn as torch_randn
from fastai.vision.all import test_eq

## DnCNN

In [1]:
#| echo: false
#| output: asis
show_doc(DnCNN)

---

[source](https://github.com/bmandracchia/bioMONAI/blob/main/bioMONAI/nets.py#L13){target="_blank" style="float:right; font-size:smaller"}

### DnCNN

>      DnCNN (channels, num_of_layers=9, features=64, kernel_size=3)

Base class for all neural network modules.

Your models should also subclass this class.

Modules can also contain other Modules, allowing to nest them in
a tree structure. You can assign the submodules as regular attributes::

    import torch.nn as nn
    import torch.nn.functional as F

    class Model(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv2d(1, 20, 5)
            self.conv2 = nn.Conv2d(20, 20, 5)

        def forward(self, x):
            x = F.relu(self.conv1(x))
            return F.relu(self.conv2(x))

Submodules assigned in this way will be registered, and will have their
parameters converted too when you call :meth:`to`, etc.

.. note::
    As per the example above, an ``__init__()`` call to the parent class
    must be made before assignment on the child.

:ivar training: Boolean represents whether this module is in training or
                evaluation mode.
:vartype training: bool

In [7]:
x = torch_randn(16, 1, 32, 64)

tst = DnCNN(1)
test_eq(tst(x).shape, x.shape)

## UNet

### Convolutional sub-unit

In [2]:
#| echo: false
#| output: asis
show_doc(SubNetConv)

---

[source](https://github.com/bmandracchia/bioMONAI/blob/main/bioMONAI/nets.py#L33){target="_blank" style="float:right; font-size:smaller"}

### SubNetConv

>      SubNetConv (ks=3, stride=1, padding=None, bias=None, ndim=2,
>                  norm_type=<NormType.Batch: 1>, bn_1st=True, act_cls=<class
>                  'torch.nn.modules.activation.ReLU'>, transpose=False,
>                  init='auto', xtra=None, bias_std=0.01, dropout=0.0)

In [9]:
x = torch_randn(16, 1, 32, 64, 64)
xdim = len(x.shape)-2

# reduce
tst = SubNetConv(3, padding=1, stride=2, ndim=xdim,
                 norm_type=NormType.Batch, dropout=.1)(1, 2, 2)
y = tst(x)
test_eq(y.shape, [16, 2, 8, 16, 16])
print(tst)
# upsample
tst = SubNetConv(ks=4, padding=0, stride=4, ndim=xdim, norm_type=NormType.Batch,
                 transpose=True)(2, 1)  # to double the size, the kernel cannot be odd
test_eq(tst(y).shape, x.shape)
del y

Sequential(
  (0): Sequential(
    (0): ConvLayer(
      (0): Conv3d(1, 2, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1), bias=False)
      (1): BatchNorm3d(2, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
    )
    (1): Dropout(p=0.1, inplace=False)
  )
  (1): Sequential(
    (0): ConvLayer(
      (0): Conv3d(2, 2, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1), bias=False)
      (1): BatchNorm3d(2, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
    )
    (1): Dropout(p=0.1, inplace=False)
  )
)


### UNet block

### Recursive UNet

In [3]:
#| echo: false
#| output: asis
show_doc(UNet)

---

[source](https://github.com/bmandracchia/bioMONAI/blob/main/bioMONAI/nets.py#L125){target="_blank" style="float:right; font-size:smaller"}

### UNet

>      UNet (depth=4, mult_chan=32, in_channels=1, out_channels=1,
>            last_activation=None, kernel_size=3, ndim=2, n_conv_per_depth=2,
>            activation='ReLU', norm_type=<NormType.Batch: 1>, dropout=0.0,
>            pool=<function MaxPool>, pool_size=2, residual=False,
>            prob_out=False, eps_scale=0.001)

Base class for all neural network modules.

Your models should also subclass this class.

Modules can also contain other Modules, allowing to nest them in
a tree structure. You can assign the submodules as regular attributes::

    import torch.nn as nn
    import torch.nn.functional as F

    class Model(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv2d(1, 20, 5)
            self.conv2 = nn.Conv2d(20, 20, 5)

        def forward(self, x):
            x = F.relu(self.conv1(x))
            return F.relu(self.conv2(x))

Submodules assigned in this way will be registered, and will have their
parameters converted too when you call :meth:`to`, etc.

.. note::
    As per the example above, an ``__init__()`` call to the parent class
    must be made before assignment on the child.

:ivar training: Boolean represents whether this module is in training or
                evaluation mode.
:vartype training: bool

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| depth | int | 4 | depth of the UNet network |
| mult_chan | int | 32 | number of filters at first layer |
| in_channels | int | 1 | number of input channels |
| out_channels | int | 1 | number of output channels |
| last_activation | NoneType | None | last activation before final result |
| kernel_size | int | 3 | kernel size of convolutional layers |
| ndim | int | 2 | number of spatial dimensions of the input data |
| n_conv_per_depth | int | 2 | number of convolutions per layer |
| activation | str | ReLU | activation function used in convolutional layers |
| norm_type | NormType | NormType.Batch |  |
| dropout | float | 0.0 |  |
| pool | function | MaxPool |  |
| pool_size | int | 2 |  |
| residual | bool | False |  |
| prob_out | bool | False |  |
| eps_scale | float | 0.001 |  |

In [13]:
x = torch_randn(16, 1, 32, 64, 64)
xdim = len(x.shape)-2

tst = UNet(depth=1, ndim=xdim, n_conv_per_depth=1, residual=True)
mods = list(tst.children())
print(mods)
test_eq(tst(x).shape, [16, 1, 32, 64, 64])

[_Net_recurse(
  (sub_conv_more): ConvLayer(
    (0): Conv3d(1, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
    (1): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (sub_u): Sequential(
    (0): MaxPool3d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (1): _Net_recurse(
      (sub_conv_more): ConvLayer(
        (0): Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
      )
    )
    (2): Upsample(scale_factor=2.0, mode='nearest')
  )
  (sub_conv_less): ConvLayer(
    (0): Conv3d(96, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
    (1): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
), ConvLayer(
  (0): Conv3d(32, 1, kernel_size=(3, 3, 3), stride=(1, 